# Week 5-6: DuckDB Feature Engineering & Leakage-Free Pipeline
**FlyRank Applied Search Intelligence ML Internship**

### Objective:
1. Use DuckDB to engineer search velocity, decay deltas, and engagement ratios.
2. Implement strict anti-leakage guards to prevent diagnostic flags from leaking ground-truth labels.
3. Establish grouped cross-validation partitioning across client domains.


In [ ]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

# Read local dataset into DuckDB virtual table
con.execute("CREATE VIEW content_raw AS SELECT * FROM read_csv_auto('../data/content_refresh_anonymized.csv')")

# Query high-level star schema aggregation
summary = con.sql("""
    SELECT 
        COUNT(*) as total_records,
        COUNT(DISTINCT client_id) as total_clients,
        AVG(impressions_90d) as avg_impressions_90d,
        AVG(clicks_90d) as avg_clicks_90d,
        AVG(avg_position) as mean_avg_position,
        AVG(ctr) as mean_ctr
    FROM content_raw
""").df()
print(summary)


## DuckDB SQL Feature Engineering
We compute 30-day velocity ratios, decay deltas, engagement density, and active impression ratios.


In [ ]:
query = """
    SELECT 
        content_id,
        client_id,
        content_age_days,
        days_since_last_update,
        search_volume,
        word_count,
        char_count,
        competition,
        cpc,
        avg_position,
        ctr,
        impressions_90d,
        clicks_90d,
        sessions_90d,
        pageviews_90d,
        -- Velocity deltas (last 30d vs prev 30d)
        (impressions_last_30d + 1.0) / (impressions_prev_30d + 1.0) AS impression_velocity,
        (clicks_last_30d + 1.0) / (clicks_prev_30d + 1.0) AS click_velocity,
        (sessions_last_30d + 1.0) / (sessions_prev_30d + 1.0) AS session_velocity,
        (impressions_prev_30d - impressions_last_30d) / (impressions_90d + 1.0) AS decay_ratio_30d,
        -- Engagement & Efficiency Ratios
        (engaged_sessions_90d * 1.0) / (sessions_90d + 1.0) AS engagement_density,
        (scroll_events_90d * 1.0) / (pageviews_90d + 1.0) AS scroll_density,
        (clicks_90d * 1.0) / (search_volume + 1.0) AS capture_efficiency,
        (days_with_impressions * 1.0) / 90.0 AS active_days_ratio,
        -- Metadata categoricals
        competition_level,
        main_intent,
        content_type,
        age_tier,
        position_tier,
        -- Ground Truth Label
        COALESCE(is_initial_refresh_candidate, FALSE)::INT AS target_refresh_opportunity
    FROM content_raw
"""
df_features = con.sql(query).df()
print(f"Engineered feature set: {df_features.shape[0]:,} rows, {df_features.shape[1]} columns.")
df_features.head(3)


## Strict Leakage Audit
We confirm that all internal diagnostic flags (`health_score`, `needs_ctr_fix`, `is_declining`, `is_quick_win`) are excluded from feature tables.


In [ ]:
leakage_flags = ['health_score', 'needs_ctr_fix', 'is_declining', 'is_quick_win', 'is_underperformer', 'trend_direction', 'trend_pct']
leaked_in_features = [col for col in leakage_flags if col in df_features.columns]
print(f"Leakage Audit Status: {'PASSED (0 leaked columns)' if not leaked_in_features else f'FAILED ({leaked_in_features})'}")


### Week 5-6 Summary:
1. Feature extraction in DuckDB is fast, memory-efficient, and cleanly modularized.
2. Velocity features capture dynamic position and impression drift over 30d/90d intervals.
3. Leakage audit ensures that test set evaluation represents realistic production conditions.
